# Laboratorio 3B: configurar un prompt

*Agentes de IA (aplicado) · Sesión 3 · Bloque 3.2 · Especialización en Inteligencia Artificial, UNAL Manizales*

GitHub crea etiquetas en cada repositorio nuevo para ordenar sus issues, como bug, enhancement, documentation y question. Etiquetar cientos de issues a mano cansa, y un modelo puede ayudar. Pero, ¿con qué prompt? Una frase de más cambia los aciertos, los tokens y el tiempo de respuesta. En este laboratorio comparas cuatro variantes de un prompt sobre los mismos issues, como un experimento.

**Conceptos clave:**
- **Variante:** una versión del prompt que cambia una sola pieza.
- **Plantilla de mensajes:** `ChatPromptTemplate` arma los mensajes de cada caso con un hueco, como `{issue}`.
- **Acierto:** la etiqueta de la respuesta coincide con la esperada.
- **Costo por mil aciertos:** lo que costaría lograr mil respuestas correctas. Es el costo total dividido por los aciertos, por mil.
- **Latencia:** los segundos que tarda cada respuesta.
- **pass^3:** la proporción de issues que aciertan en los tres intentos, de 0 a 1.
- **Cadena de pensamiento:** pedirle al modelo que razone paso a paso antes de responder.

<details>
<summary><b>Repaso: lo que viene del 2B, del 3A y del bloque 3.2</b> (clic para abrir)</summary>

- **LangChain** es la biblioteca de Python con la que el curso llama a los modelos. `init_chat_model` crea un modelo con su nombre y su proveedor.
- **Los mensajes** tienen tres roles: sistema, usuario y asistente. La API no recuerda: cada llamada lleva todo lo que el modelo debe leer.
- **`usage_metadata`** trae los tokens de entrada y de salida. Con ellos se calcula el costo.
- **Un criterio determinista** da siempre el mismo resultado con la misma respuesta.
- **pass^k**, del 2B: cada caso se repite k veces, y cuenta solo si acierta en todas.
- **La tabla de resultados** tiene las mismas columnas en todo el curso. La sección 4.2 la trae plegada.
- **La capa de evaluación** son las pruebas de cada etapa, de funcionamiento y de robustez. Se llama capa, pero no es una de las seis capas del proyecto: las acompaña a todas.
- **El razonamiento interno:** los modelos del curso pueden razonar por dentro antes de responder. Esos tokens se cobran como salida.
- **La capa 2 del proyecto** es la de las instrucciones: el prompt de sistema y el esquema de tu agente.
</details>

**En este laboratorio vas a:**
- fijar tu criterio de éxito antes de probar;
- armar cada variante como una plantilla de mensajes, y probarla sin llamar al modelo;
- correr cuatro variantes sobre los mismos doce issues, tres veces cada una;
- comparar los aciertos, los tokens, el costo por mil aciertos, la latencia y pass^3;
- en casa, comparar con el razonamiento interno y probar una variante propia.

**Resultados de aprendizaje:**
- **RA2:** diseñar prompts y salidas estructuradas, y usar la API de un LLM;
- **RA7:** probar el funcionamiento y la robustez de cada etapa, y estimar su desempeño y su costo.

**Tiempo y entrega.** Son 60 minutos en clase y unas 1.5 horas en casa. En clase haces las secciones 1 a 5, hasta el Ejercicio 3. En casa escribes el análisis (5) y haces las secciones 6 a 10, unos 70 minutos. Los otros 20 son para terminar lo que no alcances en clase. Entregas este notebook ejecutado y guardado con sus salidas, al inicio de la semana siguiente.

## Cómo usar este notebook

- **Ábrelo en VS Code.** Abre la carpeta del laboratorio con «File > Open Folder» y luego abre este archivo.
- **Elige el kernel.** Arriba a la derecha, en «Select Kernel», elige el entorno virtual del curso.
- **Ejecuta en orden,** con Shift+Enter. Cada celda usa lo que hicieron las anteriores. Si reinicias el kernel, vuelve a ejecutar desde el principio.
- **Edita solo las celdas con ✏️:** los ejercicios, la configuración y tus análisis.
- **Si ves `SyntaxError` o `NameError`,** revisa la celda ✏️ que editaste: las comillas, las comas y los paréntesis. Corrígela y ejecútala de nuevo, con las que siguen.
- **No ejecutes todo de una vez al empezar:** tu criterio del Ejercicio 1 va antes de ver los resultados.
- **Si algo falla, lee el mensaje:** dice qué revisar. Si el error sigue, reinicia el kernel y ejecuta todo de nuevo. También puedes usar el modo grabado de la sección 1.
- **Guarda con las salidas,** con Ctrl+S (Cmd+S en Mac). El docente califica sin volver a ejecutar.

<details>
<summary><b>¿No puedes instalar VS Code? Usa Google Colab como respaldo</b> (clic para abrir)</summary>

Colab sirve solo en modo grabado.
1. En Colab, abre este notebook con «Archivo > Subir notebook».
2. En el panel «Archivos», sube el archivo `lab_3B_configurar_un_prompt.zip`: el mismo .zip del laboratorio que descargaste.
3. Ejecuta la primera celda de la sección 1: descomprime el archivo y entra a su carpeta.

El archivo `LEEME.md` de la carpeta explica los pasos con más detalle.
</details>

## Contenido

* [Datos de la entrega](#0)
* [1 - Preparar el entorno](#1) · en clase, 5 min
* [2 - La tarea y el criterio de éxito](#2) · en clase, 12 min
    * [2.1 - Los issues de FotoNube](#2-1)
    * [2.2 - Ejercicio 1: tu criterio de éxito](#2-2)
    * [2.3 - Ejercicio 2: dos etiquetas esperadas](#2-3)
* [3 - Las variantes y sus plantillas](#3) · en clase, 12 min
    * [3.1 - Comparar prompts es un experimento](#3-1)
    * [3.2 - Una plantilla para cada variante](#3-2)
    * [3.3 - La construcción del prompt](#3-3)
* [4 - Las pruebas](#4) · en clase, 15 min
    * [4.1 - La cuenta antes de correr](#4-1)
    * [4.2 - La respuesta de cada variante](#4-2)
    * [4.3 - La cadena de pensamiento por dentro](#4-3)
    * [4.4 - La consistencia](#4-4)
* [5 - La comparación](#5) · en clase, 8 min; el análisis (5), para casa, 10 min
* [6 - Para casa: el razonamiento interno](#6) · para casa, 15 min
* [7 - Para casa: tu variante](#7) · para casa, 25 min
* [8 - Entregable y calificación](#8) · para casa, 5 min
* [9 - Comprobación de la entrega](#9) · para casa, 5 min
* [10 - Tu proyecto: la capa 2](#10) · para casa, 10 min
* [Conclusión](#conclusion)
* [Referencias](#referencias)

<details>
<summary><b>💡 Otra forma de navegar: el panel Esquema de VS Code</b> (clic para abrir)</summary>

VS Code muestra los títulos del notebook en el panel «Esquema» («Outline» en inglés). Lo encuentras en la barra lateral del Explorador, abajo. Haz clic en un título para saltar a esa sección. Si un enlace de esta tabla no funciona en tu versión de VS Code, usa el panel.
</details>

<a id='0'></a>
## Datos de la entrega

Escribe tu nombre, o los de la pareja si el docente lo permite. Estos datos se quedan en el notebook: nunca se envían a un modelo. El modo, el modelo y la fecha se agregan solos.

* ✏️ Escribe tu nombre en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Datos de la entrega: escribe tu nombre entre las comillas
datos_entrega = {
    "nombres": "...",           # por ejemplo, "Ana Pérez" o "Ana Pérez y Luis Gómez"
    "modalidad": "individual",  # "individual" o "pareja", según defina el docente
}

<a id='1'></a>
## 1 - Preparar el entorno

*En clase · 5 minutos*

El arranque es el mismo del Laboratorio 3A. La celda siguiente revisa la versión de Python, instala los paquetes que falten y busca los archivos del laboratorio. En Colab, antes descomprime el archivo .zip.

* Ejecuta la siguiente celda para comprobar el entorno.

In [ ]:
# Comprueba la versión de Python, los paquetes del curso y los archivos del laboratorio
import importlib.util
import subprocess
import sys
from pathlib import Path

# En Colab: si subiste el .zip del laboratorio, lo descomprime y entra a su carpeta
if "google.colab" in sys.modules and not Path("datos").exists():
    import os, zipfile
    for archivo_zip in Path("/content").glob("lab_3B*.zip"):
        zipfile.ZipFile(archivo_zip).extractall("/content")
        os.chdir("/content/lab_3B_configurar_un_prompt")

print(f"Python {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("❌ Este laboratorio necesita Python 3.11 o posterior. Elige el kernel del entorno del curso.")

# Instala en el kernel solo los paquetes que faltan, con las versiones del curso
paquetes = {"langchain": "langchain", "langchain_anthropic": "langchain-anthropic",
            "langchain_google_genai": "langchain-google-genai", "pydantic": "pydantic", "pandas": "pandas",
            "matplotlib": "matplotlib", "dotenv": "python-dotenv"}
versiones_curso = {}
if Path("requirements.txt").exists():
    for linea in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
        if "==" in linea:
            versiones_curso[linea.split("==")[0].strip()] = linea.strip()
faltan = [versiones_curso.get(pip, pip) for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalo lo que falta:", ", ".join(faltan))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=False)
    print("Listo. Reinicia el kernel y vuelve a ejecutar desde el principio.")
else:
    print("✅ Los paquetes del curso están instalados.")

# Revisa los archivos del laboratorio
for ruta in ["curso_agentes", "datos/issues_3B.json", "grabaciones"]:
    print(("✅ " if Path(ruta).exists() else "❌ falta ") + ruta)

Si ves una ❌, sigue su indicación. Si falta un archivo, abre la carpeta completa del laboratorio, no solo el notebook.

**El modo y el modelo.** Como en el 3A, el notebook corre en modo grabado, sin clave, o en vivo, con la clave del curso. El modelo por defecto es Claude Haiku 5.5. Si tu grupo eligió Gemini en la capa 1, usa Gemini 3.5 Flash-Lite: también tiene grabaciones.

> **Nota:** la clave vive en el archivo `.env`. Nunca la escribas en una celda: el notebook se entrega con todo lo que contiene.

* ✏️ Si quieres, cambia el modo, el modelo o los límites. Luego ejecuta la siguiente celda.

In [ ]:
# ✏️ Elige el modo y el modelo. Con "auto", el notebook usa el modo en vivo solo si tu clave sirve
MODO = "auto"                           # "auto", "grabado" o "en_vivo"
MODELO = "anthropic:claude-haiku-5-5"   # o "google_genai:gemini-3.5-flash-lite"
LIMITE_LLAMADAS = 300                   # en vivo, el máximo de llamadas de esta corrida
LIMITE_TOKENS = 300_000                 # en vivo, el máximo de tokens de esta corrida

import pandas as pd
import curso_agentes as ca

config = ca.configurar(modo=MODO, modelo=MODELO, max_llamadas=LIMITE_LLAMADAS, max_tokens=LIMITE_TOKENS)
ca.mostrar_ficha_entrega(datos_entrega, config)

En vivo, `init_chat_model` crea el modelo con su nombre, su proveedor y sus parámetros. En modo grabado, el módulo del curso crea un modelo simulado de LangChain que se usa igual.

* Ejecuta la siguiente celda para crear el modelo.

In [ ]:
# Crea el modelo: en vivo con init_chat_model, o el modelo simulado con sus grabaciones
from langchain.chat_models import init_chat_model

if config.modo == "en_vivo":
    modelo = init_chat_model(config.modelo, **config.parametros)
else:
    modelo = ca.modelo_grabado(config)

print("Modelo listo:", config.nombre_modelo, "·", type(modelo).__name__)
print("Parámetros:", config.parametros)
print("Versiones:", ca.versiones())

Los parámetros y las versiones quedan en las salidas, junto a tus resultados.

<details>
<summary><b>Para profundizar: los parámetros del curso</b> (clic para abrir)</summary>

Con Claude Haiku 5.5, el curso desactiva el razonamiento (`thinking`) y fija 512 tokens de salida. No envía `temperature`: Haiku 5.5 la rechaza con un error, y en Gemini 3.5 Flash-Lite es obsoleta. Por eso los tres intentos de la sección 4 varían con los valores por defecto de cada modelo. Con Gemini, el razonamiento va en su nivel mínimo, que es su valor por defecto. La sección 6 activa el razonamiento a propósito, para compararlo con la cadena de pensamiento.
</details>

<a id='2'></a>
## 2 - La tarea y el criterio de éxito

*En clase · 12 minutos*

<a id='2-1'></a>
### 2.1 - Los issues de FotoNube

**FotoNube** es la app de fotos ficticia del curso. Su repositorio recibe issues, y cada uno lleva una sola etiqueta:

| Etiqueta | En GitHub | Qué significa |
| :--- | :--- | :--- |
| error | bug | Algo no funciona como debe |
| mejora | enhancement | Una función nueva o un cambio que alguien pide |
| documentación | documentation | Algo que falta o está mal en la documentación |
| pregunta | question | Alguien pide información, sin reportar un fallo |

Son cuatro de las etiquetas que GitHub crea en cada repositorio nuevo, en español. El issue guía es el de las diapositivas: «Al girar el teléfono, el botón de subir fotos queda fuera de la pantalla.» Es un error.

* Ejecuta la siguiente celda para cargar los datos y ver los doce issues.

In [ ]:
# Carga los datos del laboratorio y muestra los doce issues, con su etiqueta esperada
null, true, false = None, True, False   # por si escribes true, false o null en minúscula, como en JSON: Python los lee como True, False y None

datos = ca.cargar_issues()
guia = next(c for c in datos["base"] if c["caso_id"] == datos["caso_guia"])
ca.mostrar_issues(datos["base"])

Ocho issues son **normales**, dos de cada etiqueta. Los otros cuatro son **difíciles** a propósito, y sirven de pruebas de robustez:
- dos casos **borde**: una pregunta que reporta un fallo (I-09) y una mejora que suena a queja (I-10);
- uno **ambiguo**, que admite dos lecturas (I-11);
- uno **mal escrito** (I-12).

La etiqueta de I-11 e I-12 la escribes tú en el Ejercicio 2. Todos los issues son ficticios.

<a id='2-2'></a>
### 2.2 - Ejercicio 1: tu criterio de éxito

**Qué se pide.** Decide, antes de probar, cuándo una variante sirve y cómo eliges entre las que sirven:
- `aciertos_minimos_pct`: el porcentaje mínimo de aciertos, sobre todos los intentos;
- `desempate`: la medida que decide entre las que sirven, `"costo_por_mil_aciertos"`, `"latencia"` o `"pass^3"`.

**Por qué.** El bloque 3.2 pide fijar el criterio antes de ver los resultados. Si lo eliges después, puedes acomodarlo a la variante que te gustó.

**Cómo saber si quedó bien.** La celda de revisión repite tu criterio en una frase.

* ✏️ Completa tu criterio en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 1: tu criterio de éxito, antes de probar
CRITERIO = {
    "aciertos_minimos_pct": ...,   # un número de 0 a 100, sin comillas; por ejemplo, 80
    "desempate": ...,              # "costo_por_mil_aciertos", "latencia" o "pass^3"
}

**✏️ Por qué ese criterio.** En una o dos oraciones: ¿para qué usaría FotoNube las etiquetas? ¿Por qué ese mínimo y ese desempate?

*Escribe aquí tu respuesta.*

* Ejecuta la siguiente celda para revisar tu criterio.

In [ ]:
# Revisa tu criterio de éxito y lo repite en una frase
criterio = ca.revisar_criterio_3B(CRITERIO)

<a id='2-3'></a>
### 2.3 - Ejercicio 2: dos etiquetas esperadas

**Qué se pide.** Escribe la etiqueta esperada de I-11 y de I-12, con las definiciones de la sección 2.1.

**Por qué.** Sin etiqueta esperada no hay acierto que medir. Un issue ambiguo admite dos lecturas, pero el experimento necesita una sola respuesta esperada. Decídela con las definiciones de la sección 2.1, antes de probar. Si dudas, la pista dice qué mirar.

**Cómo saber si quedó bien.** La celda siguiente dice «2 de 2 etiquetas del Ejercicio 2», sin pistas.

<details>
<summary><b>Pista</b> (clic para abrir)</summary>

- I-11: ¿falla la app, o falla lo que dice la documentación?
- I-12: lee más allá de la forma. ¿Qué le pasa a la búsqueda por fecha?
</details>

* ✏️ Escribe las dos etiquetas en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 2: la etiqueta esperada de I-11 y de I-12: "error", "mejora", "documentación" o "pregunta"
mis_etiquetas = {
    "I-11": ...,
    "I-12": ...,
}

* Ejecuta la siguiente celda para armar la lista de casos.

In [ ]:
# Une los doce issues con tus dos etiquetas y revisa cada una
casos = ca.armar_casos_3B(datos, mis_etiquetas, modo=config.modo)
ca.mostrar_issues([c for c in casos if c["origen"] == "tuya"])

<a id='3'></a>
## 3 - Las variantes y sus plantillas

*En clase · 12 minutos*

<a id='3-1'></a>
### 3.1 - Comparar prompts es un experimento

Comparar prompts a ojo engaña. El bloque 3.2 propone tres reglas:
- **los mismos casos:** todas las variantes corren sobre los mismos doce issues;
- **un cambio a la vez:** si cambias dos piezas, no sabes cuál ayudó;
- **el criterio, antes:** decides qué es un acierto antes de ver los resultados.

Estas son las cuatro variantes del laboratorio. Cada una agrega una sola pieza a la instrucción base.

* Ejecuta la siguiente celda para leer las cuatro variantes.

In [ ]:
# Muestra las cuatro variantes, con lo que agrega cada una y su mensaje de sistema completo
ca.mostrar_variantes(datos, ca.VARIANTES_CLASE)

Los tres ejemplos no traen ninguna «pregunta». ¿Bajan por eso los aciertos de I-07 e I-08? La sección 4 lo muestra, y en casa una variante del banco lo pone a prueba.

<a id='3-2'></a>
### 3.2 - Una plantilla para cada variante

Con doce issues y cuatro variantes, nadie escribe los mensajes a mano. Una **plantilla** los arma. `ChatPromptTemplate` recibe los mensajes con un hueco entre llaves, `{issue}`. Luego lo llena con el texto de cada caso, como en la diapositiva 7 del bloque 3.2. Así todos los issues reciben la misma instrucción, y solo cambia el issue.

* Ejecuta la siguiente celda para armar la plantilla de una variante y enviar el issue guía.

In [ ]:
# Arma la plantilla de la variante con formato explícito, y envía el issue guía
from langchain_core.prompts import ChatPromptTemplate

instruccion = ca.texto_sistema(datos, "formato")
plantilla = ChatPromptTemplate.from_messages([
    ("system", instruccion),
    ("user", "<issue>\n{issue}\n</issue>"),
])
mensajes = plantilla.invoke({"issue": guia["entrada"]})
ca.mostrar_mensajes(mensajes.to_messages())

respuesta = modelo.invoke(mensajes)
print("Respuesta:", ca.texto_de(respuesta))
print("Tokens:", respuesta.usage_metadata)

La plantilla llenó el hueco con el issue guía. Las otras variantes cambian solo el mensaje de sistema. El módulo del curso arma las cuatro plantillas de la misma forma.

* Ejecuta la siguiente celda para armar las plantillas de las cuatro variantes.

In [ ]:
# Arma una plantilla por variante, con el mismo hueco para el issue
plantillas = ca.armar_plantillas(datos, ca.VARIANTES_CLASE)
print("Plantillas listas:", ", ".join(plantillas))

<a id='3-3'></a>
### 3.3 - La construcción del prompt

La capa de evaluación del 3B tiene tres etapas:

| Etapa | Prueba de funcionamiento | Prueba de robustez | Criterio de aceptación |
| :--- | :--- | :--- | :--- |
| Construcción del prompt | La plantilla produce los mensajes esperados, sin llamar al modelo | Un ejemplo con un campo faltante | Los mensajes contienen la instrucción y los ejemplos elegidos |
| Respuesta por variante | Cada variante resuelve los casos | Casos límite de la tarea | La respuesta coincide con la esperada |
| Consistencia | Cada caso, repetido por variante | Respuestas grabadas que varían | Se reporta pass^k por variante |

La primera etapa no llama al modelo. Revisa que cada plantilla arme bien sus mensajes. Su prueba de robustez usa una lista de ejemplos con un campo faltante, a propósito: es la fila R-EJEMPLO. Ahí, «pasa» quiere decir que la prueba detectó el ejemplo incompleto.

En las otras dos etapas, los casos límite son los cuatro issues difíciles. Las respuestas que varían son los tres intentos de cada issue. En vivo cambian solas, y las grabaciones guardan tres respuestas por issue.

* Ejecuta la siguiente celda para probar la construcción de las cuatro plantillas.

In [ ]:
# Prueba la construcción de cada plantilla y una lista con un ejemplo incompleto, sin llamar al modelo
ejemplos_con_error = [
    {"texto": "La app se cierra al abrir un álbum vacío.", "etiqueta": "error"},
    {"texto": "Agregar un modo oscuro a la galería."},   # le falta la etiqueta, a propósito
]
construccion = ca.probar_construccion(datos, plantillas, guia, ejemplos_con_error, config=config)

<br>

### Análisis

Cada plantilla pasa si su mensaje de sistema trae sus piezas y el issue va al final. El ejemplo incompleto se detecta antes de gastar una llamada. Un programa que lo enviara le mostraría al modelo un ejemplo sin respuesta, y el modelo podría imitarlo.

<a id='4'></a>
## 4 - Las pruebas

*En clase · 15 minutos*

<a id='4-1'></a>
### 4.1 - La cuenta antes de correr

Antes de correr un experimento, se estima su costo, como en la diapositiva 19 del bloque 3.2. Bastan tres números: los tokens de cada llamada, cuántas llamadas y el precio. Los tokens se estiman sin llamar al modelo. Los de entrada salen del texto, con unos 3 caracteres por token. Los de salida son los de las diapositivas: unos pocos sin cadena de pensamiento, y unos 110 con ella.

* Ejecuta la siguiente celda para hacer la cuenta.

In [ ]:
# Estima las llamadas, los tokens y el costo del experimento, sin llamar al modelo
cuenta = ca.cuenta_previa(datos, plantillas, casos, k=3, config=config)

La cadena de pensamiento escribe mucho más, y con Haiku 5.5 la salida cuesta cinco veces más que la entrada. Aun así, el experimento cuesta muy poco: a lo sumo unos centavos de dólar. En producción, con miles de issues, la diferencia importa. La sección 5 muestra los tokens que contó el proveedor: compáralos con esta cuenta.

<a id='4-2'></a>
### 4.2 - La respuesta de cada variante

Ahora cada variante recibe cada issue tres veces. El primer intento mide la respuesta, y los tres juntos, la consistencia. En vivo, son 144 llamadas.

**Desglose del experimento**

| | |
| :--- | :--- |
| **Entrada** | Los doce issues, con las cuatro variantes |
| **Prueba de funcionamiento** | Cada variante resuelve los ocho issues normales |
| **Pruebas de robustez** | Los cuatro issues difíciles, de I-09 a I-12 |
| **Criterio** | La etiqueta coincide con la esperada. Una respuesta sin una sola etiqueta clara cuenta como fallo, con su razón |

* Ejecuta la siguiente celda para correr las cuatro variantes.

In [ ]:
# Envía cada issue tres veces con cada variante y aplica el criterio: la etiqueta coincide con la esperada
contenido, consistencia = ca.correr_variantes(modelo, config, plantillas, casos, k=3, datos=datos)

* Ejecuta la siguiente celda para ver qué respondió cada variante en cada issue.

In [ ]:
# Muestra la etiqueta de cada variante en cada issue, en el primer intento, y la tabla completa plegada
ca.mostrar_por_issue(contenido, casos, datos)
ca.mostrar_resultados_3B(pd.concat([construccion, contenido, consistencia], ignore_index=True))

<br>

### Análisis

Busca dónde fallan las variantes:
- **en los issues difíciles, de I-09 a I-12,** suelen separarse;
- **en I-07 e I-08, las preguntas,** mira la variante con tres ejemplos, que no trae ninguna pregunta;
- **«sin etiqueta»:** la respuesta no trajo una sola etiqueta clara;
- **«se negó»:** el modelo rechazó el prompt y no respondió.

<a id='4-3'></a>
### 4.3 - La cadena de pensamiento por dentro

La variante con cadena de pensamiento escribe su razonamiento y termina con «Etiqueta: …». El módulo lee esa última línea.

> **Nota:** Anthropic advierte que Claude Haiku 5.5 puede rechazar un prompt que le pide poner su razonamiento en la respuesta. El rechazo trae la categoría `reasoning_extraction`, y se cobra. Si pasa, el resumen de la corrida en 4.2 lo cuenta. La tabla de 4.4 lo marca con ⊘, y la cuadrícula lo suma en la columna «se negó».

* Ejecuta la siguiente celda para leer la respuesta completa de la cadena de pensamiento en I-09.

In [ ]:
# Muestra la respuesta completa de la variante con cadena de pensamiento en I-09, y la etiqueta que se leyó
ca.mostrar_respuesta_completa(contenido, "I-09", "cadena", datos)

I-09 está escrito como pregunta, pero reporta un fallo. Mira si el razonamiento lo notó, y cuántos tokens de salida costó.

<a id='4-4'></a>
### 4.4 - La consistencia

Cada issue se repitió tres veces por variante. Una celda verde acertó las tres veces, y una roja, ninguna. Una celda sin color cambió entre intentos: con esa variante, ese issue no es confiable.

* Ejecuta la siguiente celda para ver los tres intentos de cada issue.

In [ ]:
# Muestra los tres intentos de cada issue con ✓ o ✗: una fila por issue y una columna por variante
ca.mostrar_consistencia_variantes(contenido, consistencia, datos)

<a id='5'></a>
## 5 - La comparación

*En clase · 8 minutos. El análisis (5) es para casa.*

Estas son las medidas de la cuadrícula de la diapositiva 18 del bloque 3.2, ahora con tus cifras. Cuentan todos los intentos: 36 por variante, doce issues por tres.
- **% de aciertos:** los intentos con la etiqueta correcta;
- **tokens por llamada:** de entrada y de salida, como los informa el proveedor;
- **costo por mil aciertos:** lo que costaría lograr mil respuestas correctas;
- **latencia media:** los segundos por respuesta;
- **pass^3:** la proporción de issues que aciertan en los tres intentos.

* Ejecuta la siguiente celda para ver la cuadrícula, sus gráficos y lo que recomienda tu criterio.

In [ ]:
# Calcula las medidas de cada variante, las grafica y aplica tu criterio de éxito
resumen = ca.resumen_variantes(contenido, consistencia, datos)
ca.mostrar_resumen_variantes(resumen)
ca.grafico_variantes(resumen, minimo=criterio["aciertos_minimos_pct"])
recomendada = ca.recomendar(resumen, criterio)

<br>

### Análisis

Lee las medidas juntas:
- **si una variante acierta más y cuesta menos por acierto,** la decisión es fácil;
- **la cadena de pensamiento** suele costar más y tardar más. Mira si sus aciertos de más lo pagan;
- **un pass^3 bajo** quiere decir que la variante acierta a veces. Para un agente, eso es un riesgo.

Con doce issues, una diferencia de uno o dos aciertos puede ser azar.

**Ejercicio 3: la variante que eliges.** Elige la variante que usarías para etiquetar los issues de FotoNube. Puede ser la que recomienda tu criterio, u otra si tienes una razón. La celda siguiente dice si tu elección coincide con tu criterio.

* ✏️ Escribe el código de tu variante en la celda siguiente y ejecútala. Está en la cuadrícula, en la columna «código».

In [ ]:
# ✏️ Ejercicio 3: la variante que eliges: "sin_ejemplos", "tres_ejemplos", "formato" o "cadena"
VARIANTE_ELEGIDA = ...

* Ejecuta la siguiente celda para revisar tu elección.

In [ ]:
# Revisa tu elección y la compara con lo que recomienda tu criterio
eleccion = ca.revisar_eleccion(VARIANTE_ELEGIDA, recomendada, resumen)

Aquí termina la parte de clase. El análisis (5) es para casa: escríbelo antes de la sección 6, con la cuadrícula a la vista.

**✏️ Tu análisis (5), para casa.** En tres a cinco oraciones, con cifras de la cuadrícula: ¿qué variante eliges y por qué? ¿Qué pagas por ella en tokens, costo o latencia? ¿Ayudó la cadena de pensamiento?

*Escribe aquí tu análisis.*

<a id='6'></a>
## 6 - Para casa: el razonamiento interno

*Para casa · unos 15 minutos*

Los modelos de hoy pueden razonar por dentro, sin que se lo pidas. Lo vimos en la diapositiva 13 del bloque 3.2. Meincke et al. (2025) midieron qué pasa al pedir «Think step by step», en puntos porcentuales de aciertos:
- **en los modelos sin razonamiento,** el promedio subió entre 4 y 14 puntos, según el modelo. Pero con la medida estricta, acertar las 25 veces, como un pass^25, Gemini 2.0 Flash bajó 13 puntos. Y las respuestas tardaron de 35 % a 600 % más;
- **en los modelos que razonan,** cambió poco: o4-mini subió 3.1 puntos, y Gemini 2.5 Flash bajó 3.3.

Tu variante con cadena de pensamiento corrió sin razonamiento interno, como el primer grupo. Aquí corres la variante sin ejemplos con el razonamiento interno activado, sin pedirle la cadena. La diapositiva 15 lo sugiere: «Con un modelo que razona, prueba también a subir el esfuerzo». El **esfuerzo** dice cuánto puede razonar el modelo antes de responder: con más esfuerzo, más tokens de razonamiento. Lo corres con esfuerzo medio y alto. En vivo, son 72 llamadas.

> **Nota:** el curso suele correr con el razonamiento de Haiku 5.5 desactivado, y el de Gemini en su nivel mínimo. Esta sección es la excepción. La grabación guarda la respuesta final y cuántos tokens de razonamiento usó. El texto del razonamiento no se guarda.

* Ejecuta la siguiente celda para correr la variante sin ejemplos con el razonamiento activado.

In [ ]:
# Crea el modelo que razona, con esfuerzo medio y alto, y corre la variante sin ejemplos
escenarios = datos["razonamiento"]["escenarios"]   # {"razonamiento_medio": "medium", "razonamiento_alto": "high"}
if config.modo == "en_vivo":
    modelos_razonan = {nombre: init_chat_model(config.modelo, **ca.parametros_razonamiento(config.modelo, esfuerzo))
                       for nombre, esfuerzo in escenarios.items()}
else:
    modelos_razonan = {nombre: ca.modelo_grabado(config, escenario=nombre) for nombre in escenarios}

plantillas_razonan = {nombre: plantillas["sin_ejemplos"] for nombre in modelos_razonan}
contenido_razona, consistencia_razona = ca.correr_variantes(None, config, plantillas_razonan, casos, k=3,
                                                            modelos=modelos_razonan, datos=datos)

* Ejecuta la siguiente celda para comparar las dos formas de razonar.

In [ ]:
# Compara el razonamiento interno con la cadena de pensamiento y con la variante sin ejemplos
de_clase = ["sin_ejemplos", "cadena"]
comparacion_razona = ca.resumen_variantes(
    pd.concat([contenido[contenido["variante"].isin(de_clase)], contenido_razona], ignore_index=True),
    pd.concat([consistencia[consistencia["variante"].isin(de_clase)], consistencia_razona], ignore_index=True), datos)
ca.mostrar_resumen_variantes(comparacion_razona)
ca.grafico_variantes(comparacion_razona, titulo="Dos formas de razonar", minimo=criterio["aciertos_minimos_pct"])

<br>

### Análisis

Compara cuatro filas: la variante sin ejemplos, la cadena de pensamiento escrita y el razonamiento interno con dos esfuerzos. Mira los aciertos, pass^3, los tokens de salida y la latencia. El razonamiento interno no se ve en la respuesta, pero se cobra como salida.

**✏️ Tu análisis (6).** En dos a cuatro oraciones: ¿razonar por dentro acertó más que la cadena de pensamiento escrita? ¿Cuánto costó cada una en tokens de salida y en latencia? Frente a la variante sin ejemplos, ¿tu cadena se comportó como en el primer grupo de Meincke et al. (2025)?

*Escribe aquí tu análisis.*

<a id='7'></a>
## 7 - Para casa: tu variante

*Para casa · unos 25 minutos*

Como pide el trabajo independiente del bloque 3.2, diseñas una variante propia:
- primero piensa qué pieza cambiarías, respecto de cuál de las cuatro variantes, y por qué esperas que ayude;
- luego compárala con las otras cuatro, con las mismas medidas.

**Sin clave,** eliges del banco la variante más cercana a tu idea. Las seis tienen respuestas grabadas, y la tabla muestra el mensaje completo de cada una. **Con la clave del curso,** también puedes escribir tu propio mensaje de sistema.

**Ejemplos como turnos.** Una variante del banco no pone los ejemplos en el mensaje de sistema. Los pone antes del issue, como turnos de la conversación. En cada turno, el usuario escribe un issue y el asistente responde su etiqueta. Es la forma de la diapositiva 6 del bloque 3.2.

* Ejecuta la siguiente celda para ver el banco de variantes.

In [ ]:
# Muestra las seis variantes del banco: de cuál parte cada una, qué pieza cambia y su mensaje completo
ca.mostrar_banco_variantes(datos)

**Qué se pide.** Elige tu variante y escribe por qué esperas que ayude. Por ejemplo, «cuatro_ejemplos» parte de la variante con tres ejemplos y agrega un ejemplo de «pregunta».

**Por qué.** Un cambio a la vez te dice qué pieza ayudó, o cuál estorbó.

**Cómo saber si quedó bien.** La celda siguiente nombra tu variante y la compara con las cuatro de clase. También le aplica tu criterio del Ejercicio 1.

<details>
<summary><b>Con la clave del curso: un mensaje de sistema propio</b> (clic para abrir)</summary>

En vivo, escribe en `MI_TEXTO` el mensaje de sistema completo de tu variante. Parte del texto de una de las cuatro variantes, que viste en la sección 3.1, y cambia una sola pieza. El issue se agrega solo, al final.
</details>

* ✏️ Completa tu variante en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 4: tu variante. Sin clave, un código del banco; por ejemplo, "definiciones"
MI_VARIANTE = ...
MI_TEXTO = None       # en vivo, si quieres: tu mensaje de sistema completo, entre triples comillas, como POR_QUE
POR_QUE = """..."""   # qué pieza cambiaste, respecto de cuál variante, y por qué esperas que ayude
# Entre las triples comillas puedes escribir varias líneas. Para citar algo dentro del texto, usa « »

* Ejecuta la siguiente celda para correr tu variante y compararla. En vivo, son 36 llamadas.

In [ ]:
# Revisa tu variante, la corre sobre los mismos issues, la compara con las de clase y le aplica tu criterio
codigo_mio, plantilla_mia = None, None
contenido_mia, consistencia_mia = pd.DataFrame(), pd.DataFrame()
codigo_mio, plantilla_mia = ca.revisar_mi_variante(datos, MI_VARIANTE, MI_TEXTO, config, POR_QUE)
if plantilla_mia is not None:
    contenido_mia, consistencia_mia = ca.correr_variantes(modelo, config, {codigo_mio: plantilla_mia}, casos, k=3, datos=datos)
    todas = ca.resumen_variantes(pd.concat([contenido, contenido_mia], ignore_index=True),
                                 pd.concat([consistencia, consistencia_mia], ignore_index=True), datos)
    ca.mostrar_resumen_variantes(todas)
    ca.grafico_variantes(todas, titulo="Tu variante y las cuatro de clase", minimo=criterio["aciertos_minimos_pct"])
    ca.juzgar_mi_variante(todas, criterio, codigo_mio)

* Ejecuta la siguiente celda para ver tu variante junto a la variante de la que parte, issue por issue.

In [ ]:
# Tu variante junto a la de la que parte: la etiqueta de cada issue y sus tres intentos
if plantilla_mia is not None:
    parte_de = datos["variantes"].get(codigo_mio, {}).get("parte_de")
    vecinas = [parte_de] if parte_de else ca.VARIANTES_CLASE
    vecinas_y_mia = pd.concat([contenido[contenido["variante"].isin(vecinas)], contenido_mia], ignore_index=True)
    ca.mostrar_por_issue(vecinas_y_mia, casos, datos)
    ca.mostrar_consistencia_variantes(
        vecinas_y_mia, pd.concat([consistencia[consistencia["variante"].isin(vecinas)], consistencia_mia], ignore_index=True), datos)

**✏️ Tu análisis (7).** En tres a cinco oraciones, con cifras: ¿tu variante ayudó frente a la variante de la que parte? ¿En qué issues cambió la etiqueta? ¿La usarías, con tu criterio del Ejercicio 1?

*Escribe aquí tu análisis.*

<a id='8'></a>
## 8 - Entregable y calificación

*Para casa · 5 minutos*

**Qué entregas.** Este notebook, ejecutado de principio a fin y guardado con sus salidas. Debe tener:
- tus datos de la entrega;
- los cuatro ejercicios resueltos, con sus tablas y sus gráficos;
- el porqué de tu criterio y tus análisis, en las celdas ✏️;
- la declaración de uso de asistentes de IA y la comprobación de la entrega.

**Cómo se califica.** El laboratorio vale el 2.5 % de la calificación del curso:

| Criterio | Peso | Qué mira el docente |
| :--- | :--- | :--- |
| Funciona según su criterio de éxito | 40 % | El notebook corre de principio a fin sin errores, y tu elección se apoya en tu criterio del Ejercicio 1 |
| Las pruebas están completas y corridas | 40 % | Las tres etapas, con tus etiquetas; la comparación de las variantes; y las corridas de casa |
| El análisis está al día | 20 % | Tus análisis citan las cifras de esta corrida. También cuentan el porqué de tu criterio y la sección «Tu proyecto» |

«Al día» quiere decir que lo que escribes coincide con las salidas guardadas. Si vuelves a ejecutar y las cifras cambian, actualiza tus análisis.

**Cómo guardar con las salidas.**
1. Ejecuta todo en orden, hasta la comprobación de la sección 9.
2. Guarda con Ctrl+S (Cmd+S en Mac).
3. Cierra el notebook y vuelve a abrirlo. Si ves las tablas y los gráficos sin ejecutar nada, quedaron guardados.

> **Nota:** no uses «Clear All Outputs» antes de entregar. Esa opción borra las salidas que el docente necesita ver.

**Uso de asistentes de IA.** El curso permite usar asistentes de IA como apoyo, siempre que declares su uso.

* ✏️ Completa la declaración en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Declaración de uso de asistentes de IA
declaracion_ia = {
    "usaste_asistentes": None,   # True o False, sin comillas
    "cuales_y_para_que": "...",  # por ejemplo: "Claude, para entender qué hace ChatPromptTemplate"
}

<a id='9'></a>
## 9 - Comprobación de la entrega

*Para casa · 5 minutos*

La celda siguiente revisa que la entrega esté completa. Muestra «listo», «revisa» o «falta» en cada parte. «Revisa» quiere decir que la parte está, pero tiene algo por corregir. Orienta, no califica. Como no puede leer las celdas de texto, tus análisis los revisas tú. La sección 10, «Tu proyecto», viene después: complétala antes de entregar.

* Ejecuta la siguiente celda para comprobar tu entrega.

In [ ]:
# Revisa que la entrega esté completa: muestra «listo», «revisa» o «falta» en cada parte
# Si una celda anterior falló, esa parte sale como «falta», y esta celda no se cae
leer = globals().get   # lee un nombre del notebook; si no existe, da None
entrega_completa = ca.comprobar_entrega_3B(
    datos_entrega=leer("datos_entrega"), config=leer("config"), criterio=leer("CRITERIO"), casos=leer("casos"),
    construccion=leer("construccion"), contenido=leer("contenido"), consistencia=leer("consistencia"),
    eleccion=leer("eleccion"), contenido_razonamiento=leer("contenido_razona"), contenido_mia=leer("contenido_mia"),
    por_que=leer("POR_QUE"), declaracion_ia=leer("declaracion_ia"), variante_elegida=leer("VARIANTE_ELEGIDA"),
)

<a id='10'></a>
## 10 - Tu proyecto: la capa 2

*Para casa · 10 minutos*

**Qué capa avanza.** Este laboratorio completa la capa 2 de tu proyecto: las instrucciones. Su resultado va en la sección 2 de la ficha del agente, que se entrega al inicio del bloque 4.1.

**Cómo reutilizar este notebook.** Cambia los issues de `datos/issues_3B.json` por los diez casos de la capa 1, con su respuesta esperada. Un asistente de código puede ayudarte a adaptarlo, como sugiere el trabajo independiente. Arma dos o tres variantes de tu prompt de sistema, con una sola pieza distinta entre ellas. Fija tu criterio antes de probar, y corre cada variante tres veces. Por ejemplo:
- si tu agente **clasifica**, prueba con y sin ejemplos, y con una definición de cada categoría;
- si **extrae datos**, prueba con y sin un ejemplo de registro completo;
- si **resume**, prueba con y sin un formato explícito de salida;
- si **responde preguntas**, prueba si pedir una explicación corta mejora los aciertos.

**Qué llevas a la ficha.** El prompt de sistema que elegiste, con sus cifras: los aciertos, el costo por mil aciertos, la latencia y pass^3. Usa datos sintéticos o anonimizados, nunca datos personales reales.

**Qué sigue.** La capa 3, en la sesión 4, pone tu llamada dentro del ciclo de un agente.

**✏️ Tu proyecto.** En dos o tres oraciones: ¿qué variantes de tu prompt de sistema vas a comparar? ¿Qué pieza cambia entre ellas, y con qué criterio vas a elegir?

*Escribe aquí tu respuesta.*

<a id='conclusion'></a>
## Conclusión

En este laboratorio comparaste variantes de un prompt como un experimento. Usaste los mismos casos, un cambio a la vez y el criterio antes. Armaste cada variante con una plantilla y probaste su construcción sin llamar al modelo. Mediste los aciertos, los tokens, el costo por mil aciertos, la latencia y la consistencia. Viste que la cadena de pensamiento cuesta más, y que los modelos de hoy también razonan por dentro.

Con esto cierras la sesión 3 y la capa 2. La sesión 4 pasa de los LLMs a los agentes.

<a id='referencias'></a>
## Referencias

**Del bloque 3.2**
- Meincke, L., Mollick, E., Mollick, L. y Shapiro, D. (2025, 8 de junio). *Prompting Science Report 2: The Decreasing Value of Chain of Thought in Prompting*. Wharton Generative AI Labs. https://gail.wharton.upenn.edu/research-and-insights/tech-report-chain-of-thought/
- Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., Chi, E., Le, Q. y Zhou, D. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS 2022. https://arxiv.org/abs/2201.11903
- Anthropic. *Working with messages*. https://platform.claude.com/docs/en/build-with-claude/working-with-messages
- Anthropic. *Effort*. https://platform.claude.com/docs/en/build-with-claude/effort
- Anthropic. *Refusals and fallback*. https://platform.claude.com/docs/en/build-with-claude/refusals-and-fallback
- Google. *Gemini thinking*. https://aistudio.google.com/docs/thinking?generate-content=true
- GitHub. *Managing labels*. https://docs.github.com/en/issues/using-labels-and-milestones-to-track-work/managing-labels

**De la capa de evaluación del curso (Anexo J)**
- Grace, M., Hadfield, J., Olivares, R. y De Jonghe, J. (2026, 9 de enero). *Demystifying evals for AI agents*. Anthropic. https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents
- Yao, S., Shinn, N., Razavi, P. y Narasimhan, K. (2025). *τ-bench: A Benchmark for Tool-Agent-User Interaction in Real-World Domains*. ICLR 2025. https://arxiv.org/abs/2406.12045

**Documentación de las herramientas**, consultada el 24 de noviembre de 2026
- LangChain. *Models*: `init_chat_model` y los mensajes. https://docs.langchain.com/oss/python/langchain/models
- LangChain. *Unit testing*: los modelos simulados. https://docs.langchain.com/oss/python/langchain/test/unit-testing
- Visual Studio Code. *Jupyter Notebooks in VS Code*. https://code.visualstudio.com/docs/datascience/jupyter-notebooks